In [3]:
import pandas as pd
import math

df = pd.read_csv("477-lab-exam.csv")

print(df.head())

   Record_ID  Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  \
0          1      47.52     496.00      44.74      51.11      51.41   
1          2      37.90      42.16      42.16      41.12     426.00   
2          3      61.16      55.14      57.20      62.94      59.20   
3          4      55.57      54.20      46.56      50.81      51.37   
4          5      58.94      60.95      59.43     591.00      56.81   

   Feature_6  Feature_7  Feature_8  Feature_9  ...  Feature_12  Feature_13  \
0      49.93      49.83      45.52      50.95  ...       51.49       56.14   
1      42.74      44.62      44.79     519.00  ...       44.26       61.28   
2      51.24      45.55      57.14      50.96  ...       49.51       56.43   
3      51.18      50.21      56.32      46.84  ...       41.49       48.29   
4      51.92     527.00      56.32      50.46  ...       56.24       51.95   

   Feature_14  Feature_15  Feature_16  Feature_17  Feature_18  Feature_19  \
0       51.67       56.89  

In [4]:
X = df.drop(columns=["Record_ID", "Class"], errors="ignore")
Y = df["Class"]

features = list(X.columns)
n = len(Y)

In [37]:

def pearson(x, y):
    n = len(x)
    x_mean = sum(x) / n
    y_mean = sum(y) / n

    numerator = 0
    x_sum = 0
    y_sum = 0

    for i in range(n):
        dx = x[i] - x_mean
        dy = y[i] - y_mean
        numerator += dx * dy
        x_sum += dx ** 2
        y_sum += dy ** 2

    denominator = math.sqrt(x_sum * y_sum)


    if denominator == 0:
        return 0

    return numerator / denominator

pearson_opt = {}
for feature in features:
    x = list(X[feature])
    y = list(Y)
    r = pearson(x, y)
    pearson_opt[feature] = abs(r)

sorted_p = sorted(pearson_opt.items(),
                    key=lambda x: x[1], reverse=True)

for f, s in sorted_p:
    print(f, ": ", round(s, 5))

Feature_8 :  0.762
Feature_20 :  0.3402
Feature_11 :  0.33208
Feature_13 :  0.29855
Feature_1 :  0.24037
Feature_4 :  0.19577
Feature_15 :  0.19485
Feature_16 :  0.17072
Feature_6 :  0.16646
Feature_7 :  0.15286
Feature_12 :  0.14472
Feature_3 :  0.12951
Feature_19 :  0.11965
Feature_2 :  0.08538
Feature_5 :  0.08255
Feature_17 :  0.07319
Feature_10 :  0.04312
Feature_14 :  0.03346
Feature_9 :  0.00845
Feature_18 :  0.00515


In [39]:
classes = list(set(Y))
print(classes)

[1, 2, 3, 4, 5]


In [44]:

def anova_f(x, y):
    classes = list(set(y))
    total = len(x)
    grand_mean = sum(x) / total

    ssb = 0
    ssw = 0

    for c in classes:
        group = []
        for i in range(total):
            if y[i] == c:
                group.append(x[i])

        group_mean = sum(group) / len(group)
        ssb += len(group) * (group_mean - grand_mean) ** 2
        
        for v in group:
            ssw += (v - group_mean) ** 2

    msb = ssb / (len(classes) - 1)
    msw = ssw / (total - len(classes))

    if msw == 0:
        return 0

    return msb / msw

anova_opt = {}

for feature in features:
    x = list(X[feature])
    y = list(Y)
    f = anova_f(x, y)
    anova_opt[feature] = f

anova_rank = sorted(anova_opt.items(), key=lambda x: x[1], reverse=True)

for a, b in anova_rank:
    print(a, ": ", round(b, 5))


Feature_8 :  17.17696
Feature_17 :  3.56703
Feature_11 :  2.7348
Feature_13 :  2.23213
Feature_20 :  2.09317
Feature_3 :  1.6782
Feature_15 :  1.51501
Feature_16 :  1.51002
Feature_6 :  1.45245
Feature_4 :  1.40227
Feature_1 :  1.39004
Feature_10 :  1.05836
Feature_2 :  1.03136
Feature_12 :  0.97378
Feature_18 :  0.87447
Feature_14 :  0.72283
Feature_5 :  0.71809
Feature_19 :  0.61981
Feature_7 :  0.61411
Feature_9 :  0.06931


In [48]:
### entropy problem

# Convert continuous values into integers
for feature in X.columns:
    X[feature] = X[feature].astype(int)


# ==========================================
# 2. ENTROPY FUNCTION
# ==========================================

def entropy(data):

    total = len(data)
    counts = {}

    for value in data:
        if value not in counts:
            counts[value] = 0
        counts[value] += 1

    result = 0

    for value in counts:
        p = counts[value] / total
        result -= p * math.log2(p)

    return result


# ==========================================
# 3. CALCULATE PARENT ENTROPY
# ==========================================

parent_entropy = entropy(Y)

print("Parent Entropy:", parent_entropy)


# ==========================================
# 4. CALCULATE INFORMATION GAIN
# ==========================================

IG = {}

for feature in X.columns:

    weighted_entropy = 0

    values = X[feature].unique()

    for value in values:

        subset = Y[X[feature] == value]

        weight = len(subset) / len(Y)

        weighted_entropy += weight * entropy(subset)

    information_gain = parent_entropy - weighted_entropy

    IG[feature] = information_gain

    print(feature, "Information Gain:", information_gain)


# ==========================================
# 5. RANK FEATURES
# ==========================================

ranked_features = sorted(
    IG.items(),
    key=lambda item: item[1],
    reverse=True
)

print("\n========== FEATURE RANKING ==========")

for feature, score in ranked_features:
    print(feature, ":", score)


# ==========================================
# 6. SELECT TOP 10 FEATURES
# ==========================================

top_10 = ranked_features[:10]

print("\n========== TOP 10 FEATURES ==========")

for feature, score in top_10:
    print(feature, ":", score)

top_10_features = [feature for feature, score in top_10]

print("\nSelected Features:", top_10_features)


Parent Entropy: 2.321928094887362
Feature_1 Information Gain: 1.996634844757554
Feature_2 Information Gain: 1.809735285398626
Feature_3 Information Gain: 1.8570258449306316
Feature_4 Information Gain: 1.7591487851390097
Feature_5 Information Gain: 1.6162438445844767
Feature_6 Information Gain: 1.6917325948008235
Feature_7 Information Gain: 1.0909505944546685
Feature_8 Information Gain: 1.4215370947142847
Feature_9 Information Gain: 1.3591487851390096
Feature_10 Information Gain: 1.511341594627746
Feature_11 Information Gain: 1.524442035268818
Feature_12 Information Gain: 1.4615370947142847
Feature_13 Information Gain: 1.1440510350957405
Feature_14 Information Gain: 1.334246535182279
Feature_15 Information Gain: 1.174246535182279
Feature_16 Information Gain: 1.2844420352688177
Feature_17 Information Gain: 1.4888984057992976
Feature_18 Information Gain: 1.7664393446710154
Feature_19 Information Gain: 1.4044420352688176
Feature_20 Information Gain: 1.4464393446710155

========== FEATURE R

In [49]:

# Convert float values to integers
X = X.astype(int)

# Entropy function
def entropy(y):
    counts = y.value_counts()
    total = len(y)
    return -sum((c/total) * math.log2(c/total) for c in counts)

# Parent entropy
H = entropy(Y)

# Information Gain
IG = {}

for col in X.columns:
    weighted = 0

    for val in X[col].unique():
        subset = Y[X[col] == val]
        weighted += (len(subset)/len(Y)) * entropy(subset)

    IG[col] = H - weighted

# Print top 10
top10 = sorted(IG, key=IG.get, reverse=True)[:10]

print("Parent Entropy:", H)
print("\nTop 10 Features:")

for f in top10:
    print(f, IG[f])


Parent Entropy: 2.321928094887362

Top 10 Features:
Feature_1 1.996634844757554
Feature_3 1.8570258449306316
Feature_2 1.809735285398626
Feature_18 1.7664393446710154
Feature_4 1.7591487851390097
Feature_6 1.6917325948008235
Feature_5 1.6162438445844767
Feature_11 1.524442035268818
Feature_10 1.511341594627746
Feature_17 1.4888984057992976
